# CS7641 — Optimization & Learning: Unified Notebook
This single notebook contains everything for:
- **Part 1:** Randomized Optimization (RHC, SA, GA) on the **last layers** (≤ ~50k tail params)
- **Part 2:** Optimizer Ablations (SGD, Momentum, Nesterov, Adam variants, AdamW) on the **full network**
- **Part 3:** Regularization (Adam only): L2, Early Stopping, Dropout, Label Smoothing / Target Noise, Input Noise

> **Before running:** put your datasets here relative to this notebook:
>
> - `./data/hotel_bookings.csv`
> - `./data/US_Accidents_March23.csv`


## 0) Setup & Preprocessing

In [ ]:

import os, sys, math, time, json, numpy as np, pandas as pd
from pathlib import Path

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Paths
DATA_DIR = Path("./data")
HOTEL_CSV = DATA_DIR / "hotel_bookings.csv"
ACCIDENTS_CSV = DATA_DIR / "US_Accidents_March23.csv"
OUT_DIR = Path("./outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# Seed
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED); torch.manual_seed(GLOBAL_SEED)

print("Python:", sys.version.split()[0], "| Torch:", torch.__version__)


In [ ]:

# Preprocessing helpers (classification & regression)
def build_preprocessor(X: pd.DataFrame):
    num = X.select_dtypes(include=["number"]).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    pre = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat)
    ], remainder="drop", sparse_threshold=0)
    return pre

def build_preprocessor_regression(X: pd.DataFrame):
    # same for tabular regression
    return build_preprocessor(X)

def load_hotel(path, sample_size=None):
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Place hotel_bookings.csv in ./data")
    return pd.read_csv(path, nrows=sample_size)

def load_accidents(path, sample_size=None):
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Place US_Accidents_March23.csv in ./data")
    return pd.read_csv(path, nrows=sample_size)

def hotel_apply_leakage_controls(df: pd.DataFrame) -> pd.DataFrame:
    # Remove potential leakage columns (reservation outcome info)
    drop_cols = [c for c in df.columns if "reservation_status" in c.lower() or "reservation_status_date" in c.lower()]
    return df.drop(columns=drop_cols, errors="ignore")

def accidents_apply_leakage_controls(df: pd.DataFrame) -> pd.DataFrame:
    # Placeholder for potential post-outcome columns
    return df


## 1) Backbone MLP (fixed across all parts)

In [ ]:

class MLP(nn.Module):
    def __init__(self, input_dim, hidden=[128,64], out_dim=2, activation="relu", dropout_p=0.3):
        super().__init__()
        act = nn.ReLU() if activation=="relu" else nn.Tanh()
        layers=[]; prev=input_dim
        for h in hidden:
            layers += [nn.Linear(prev,h), act, nn.Dropout(dropout_p)]; prev=h
        layers += [nn.Linear(prev,out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


## 2) Dataset Preparation (Hotel / Accidents)

In [ ]:

def prepare_hotel():
    df = load_hotel(HOTEL_CSV)
    df = hotel_apply_leakage_controls(df)
    df = df.dropna(subset=["is_canceled"]).copy()
    y = df["is_canceled"].astype(int).values
    X = df.drop(columns=["is_canceled"])
    pre = build_preprocessor(X)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=GLOBAL_SEED)
    Xtr = pre.fit_transform(Xtr).astype("float32")
    Xte = pre.transform(Xte).astype("float32")
    return (Xtr, ytr), (Xte, yte)

def prepare_accidents():
    df = load_accidents(ACCIDENTS_CSV)
    df = accidents_apply_leakage_controls(df)
    df = df.dropna(subset=["Severity"]).copy()
    y = df["Severity"].astype(float).values
    X = df.drop(columns=["Severity"])
    pre = build_preprocessor_regression(X)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=GLOBAL_SEED)
    Xtr = pre.fit_transform(Xtr).astype("float32")
    Xte = pre.transform(Xte).astype("float32")
    return (Xtr, ytr), (Xte, yte)


## Part 1 — Randomized Optimization on the last layers

In [ ]:

# Tail helpers for RO
def freeze_all_but_last_k(model: nn.Module, k: int = 2):
    mods = [m for m in model.modules() if hasattr(m, "weight") and isinstance(getattr(m, "weight"), torch.nn.Parameter)]
    keep = set(mods[-k:]) if k>0 else set()
    for m in mods:
        req = (m in keep)
        m.weight.requires_grad_(req)
        if getattr(m, "bias", None) is not None:
            m.bias.requires_grad_(req)
    return [p for p in model.parameters() if p.requires_grad]

def vectorize_params(params): return torch.cat([p.detach().flatten() for p in params])
def devectorize_params(params, vec):
    off=0
    for p in params:
        n=p.numel(); p.data.copy_(vec[off:off+n].view_as(p)); off+=n

class EvalState:
    def __init__(self): self.fn_evals=0; self.best=float("inf"); self.best_vec=None; self.hist=[]

def make_objective_fn(model, loss_fn, X_val, y_val, batch_size=2048, device="cpu"):
    model.eval()
    Xv = torch.tensor(X_val.astype("float32")).to(device)
    y_is_float = y_val.dtype.kind in {"f"}
    yv = torch.tensor(y_val.astype("float32" if y_is_float else "int64")).to(device)
    dl = DataLoader(TensorDataset(Xv,yv), batch_size=batch_size, shuffle=False)
    state = EvalState()
    @torch.no_grad()
    def f(theta):
        t0=time.time()
        tail=[p for p in model.parameters() if p.requires_grad]
        devectorize_params(tail, theta)
        tot=0.0; cnt=0
        for xb,yb in dl:
            out=model(xb)
            if isinstance(loss_fn, nn.CrossEntropyLoss):
                loss = loss_fn(out, yb)
            else:
                loss = loss_fn(out.squeeze(), yb)
            tot += float(loss.item())*len(xb); cnt+=len(xb)
        val = tot/max(cnt,1); state.fn_evals+=1
        if val < state.best: state.best=val; state.best_vec=theta.clone()
        state.hist.append((state.fn_evals, val, time.time()-t0))
        return val
    f.state=state
    return f


In [ ]:

# RO algorithms (RHC, SA, GA)
def _perturb(vec, sigma, rng):
    return vec + torch.from_numpy(rng.normal(0.0, sigma, size=vec.shape).astype(np.float32))

def rhc(objective, x0, budget=20000, restarts=10, sigma_init=0.02, cool_every=200, sigma_cool=0.9, sigma_floor=0.002, reject_boost_after=100, boost_factor=1.5, seed=42):
    rng=np.random.default_rng(seed); best=None; trace=[]; per=max(budget//restarts,1); evals0=objective.state.fn_evals
    for _ in range(restarts):
        x=x0.clone(); val=objective(x)
        sigma=sigma_init*(x.std().item() if x.std().item()>0 else 1.0); rejects=0
        for _ in range(per-1):
            if (objective.state.fn_evals - evals0) % cool_every == 0: sigma=max(sigma_floor, sigma_cool*sigma)
            xp=_perturb(x,sigma,rng); vp=objective(xp)
            if vp<val: x,val=xp,vp; rejects=0
            else:
                rejects+=1
                if rejects>=reject_boost_after: sigma*=boost_factor; rejects=0
            trace.append((objective.state.fn_evals, float(val)))
        cand=(val,x.clone())
        if best is None or cand[0]<best[0]: best=cand
    return {"name":"RHC","best_val":float(best[0]),"best_x":best[1],"trace":trace,"evals":(objective.state.fn_evals-evals0)}

def sa(objective, x0, budget=20000, sigma_init=0.02, T0_scale=0.1, gamma=0.995, cool_every=200, sigma_cool=0.9, sigma_floor=0.002, seed=42):
    rng=np.random.default_rng(seed); x=x0.clone(); val=objective(x)
    sigma=sigma_init*(x.std().item() if x.std().item()>0 else 1.0); T0=T0_scale*max(val,1e-8); T=T0
    trace=[(objective.state.fn_evals,float(val))]; evals0=objective.state.fn_evals
    while (objective.state.fn_evals - evals0) < budget-1:
        xp=_perturb(x,sigma,rng); vp=objective(xp)
        if (vp<val) or (rng.random() < math.exp(-(vp-val)/max(T,1e-12))): x,val=xp,vp
        T*=gamma
        if (objective.state.fn_evals - evals0) % cool_every == 0: sigma=max(sigma_floor, sigma_cool*sigma)
        trace.append((objective.state.fn_evals,float(val)))
    return {"name":"SA","best_val":float(val),"best_x":x.clone(),"trace":trace,"evals":(objective.state.fn_evals-evals0)}

def ga(objective, x0, budget=20000, pop_size=64, elite=2, p_crossover=0.5, mut_scale=0.5, sigma_init=0.02, seed=42):
    rng=np.random.default_rng(seed); d=x0.numel()
    sigma0=sigma_init*(x0.std().item() if x0.std().item()>0 else 1.0)
    P=[_perturb(x0,sigma0,rng) for _ in range(pop_size)]
    vals=[objective(p) for p in P]; trace=[(objective.state.fn_evals,float(min(vals)))]; evals0=objective.state.fn_evals
    def tournament():
        a,b,c=rng.integers(0,pop_size,3); return P[min([a,b,c], key=lambda i: vals[i])]
    while (objective.state.fn_evals - evals0) + pop_size <= budget:
        children=[]; elite_idx=np.argsort(vals)[:elite]
        for ei in elite_idx: children.append(P[ei].clone())
        while len(children)<pop_size:
            p1,p2=tournament().clone(), tournament().clone()
            mask=torch.from_numpy(rng.random(d)<p_crossover).bool()
            child=p1.clone(); child[mask]=p2[mask]
            child=child + torch.from_numpy(rng.normal(0, mut_scale*sigma0, d).astype(np.float32))
            children.append(child)
        P=children; vals=[objective(p) for p in P]
        trace.append((objective.state.fn_evals,float(min(vals))))
    bi=int(np.argmin(vals))
    return {"name":"GA","best_val":float(vals[bi]),"best_x":P[bi].clone(),"trace":trace,"evals":(objective.state.fn_evals-evals0)}


In [ ]:

# ▶️ Run Part 1
results_part1 = {}
for name, prep, classification in [
    ("hotel", prepare_hotel, True),
    ("accidents", prepare_accidents, False)
]:
    try:
        (Xtr,ytr),(Xva,yva) = prep()
    except FileNotFoundError as e:
        print(f"Skipping {name}: {e}")
        continue
    in_dim = Xtr.shape[1]
    model = MLP(in_dim, [128,64], out_dim=(2 if classification else 1))
    tail = freeze_all_but_last_k(model, k=2)
    tail_params = sum(p.numel() for p in tail)
    print(f"[{name}] tail param count:", tail_params)
    assert tail_params <= 50000, "Tail exceeds ~50k; reduce hidden dims."

    loss_fn = nn.CrossEntropyLoss() if classification else nn.MSELoss()
    objective = make_objective_fn(model, loss_fn, Xva, yva, batch_size=2048)

    x0 = vectorize_params(tail)
    budget = 20000 if name == "hotel" else 30000

    R = rhc(objective, x0, budget=budget)
    S = sa(objective, x0, budget=budget)
    G = ga(objective, x0, budget=budget)
    results_part1[name] = {"RHC": R["best_val"], "SA": S["best_val"], "GA": G["best_val"]}
    print(name, results_part1[name])
results_part1


## Part 2 — Optimizer Ablations on the full network

In [ ]:

def train_one(model, Xtr, ytr, Xva, yva, classification: bool, optimizer_name: str, lr=1e-3, beta1=0.9, beta2=0.999, epochs=20, batch_size=512, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr_t = torch.tensor(Xtr.astype("float32")); Xva_t = torch.tensor(Xva.astype("float32"))
    if classification:
        ytr_t = torch.tensor(ytr.astype("int64")); yva_t = torch.tensor(yva.astype("int64"))
        crit = nn.CrossEntropyLoss()
    else:
        ytr_t = torch.tensor(ytr.astype("float32")); yva_t = torch.tensor(yva.astype("float32"))
        crit = nn.MSELoss()
    dl = DataLoader(TensorDataset(Xtr_t,ytr_t), batch_size=batch_size, shuffle=True)

    if optimizer_name=="sgd":
        opt = optim.SGD(model.parameters(), lr=lr)
    elif optimizer_name=="sgd_momentum":
        opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_name=="nesterov":
        opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)
    elif optimizer_name=="adam":
        opt = optim.Adam(model.parameters(), lr=lr, betas=(beta1,beta2))
    elif optimizer_name=="adam_nobias":
        class AdamNoBias(optim.Optimizer):
            def __init__(self, params, lr=1e-3, betas=(0.9,0.999), eps=1e-8):
                super().__init__(params, dict(lr=lr, betas=betas, eps=eps))
            def step(self, closure=None):
                loss=None
                if closure is not None: loss=closure()
                for group in self.param_groups:
                    beta1,beta2=group["betas"]; eps=group["eps"]
                    for p in group["params"]:
                        if p.grad is None: continue
                        g = p.grad.detach()
                        st = self.state[p]
                        if len(st)==0:
                            st["m"]=torch.zeros_like(p); st["v"]=torch.zeros_like(p)
                        m,v = st["m"], st["v"]
                        m.mul_(beta1).add_(g, alpha=1-beta1)
                        v.mul_(beta2).addcmul_(g,g, value=1-beta2)
                        p.addcdiv_(m, v.sqrt().add_(eps), value=-group["lr"])
                return loss
        opt = AdamNoBias(model.parameters(), lr=lr, betas=(beta1,beta2))
    elif optimizer_name=="adam_beta1zero":
        opt = optim.Adam(model.parameters(), lr=lr, betas=(0.0,beta2))
    elif optimizer_name=="adamw":
        opt = optim.AdamW(model.parameters(), lr=lr, betas=(beta1,beta2), weight_decay=1e-4)
    else:
        raise ValueError("unknown optimizer")

    hist={"train_loss":[], "val_loss":[]}
    for ep in range(epochs):
        model.train(); tot=0; n=0
        for xb,yb in dl:
            opt.zero_grad()
            out = model(xb)
            loss = crit(out if classification else out.squeeze(), yb)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += float(loss.item()); n += 1
        tr = tot/max(n,1)
        model.eval()
        with torch.no_grad():
            out = model(Xva_t)
            vl = float(crit(out if classification else out.squeeze(), yva_t).item())
        hist["train_loss"].append(tr); hist["val_loss"].append(vl)
    return hist


In [ ]:

def coarse_grids(model_fn, Xtr, ytr, Xte, yte, classification):
    alphas = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
    beta1s = [0.0, 0.5, 0.9, 0.95, 0.99]
    beta2s = [0.9, 0.95, 0.99, 0.995]
    heat1, heat2 = {}, {}
    for a in alphas:
        for b1 in beta1s:
            m = model_fn(); h = train_one(m, Xtr, ytr, Xte, yte, classification, "adam", lr=a, beta1=b1, epochs=10)
            heat1[str((a,b1))] = min(h["val_loss"])
    for a in alphas:
        for b2 in beta2s:
            m = model_fn(); h = train_one(m, Xtr, ytr, Xte, yte, classification, "adam", lr=a, beta2=b2, epochs=10)
            heat2[str((a,b2))] = min(h["val_loss"])
    return heat1, heat2

def stability_and_threshold(model_fn, Xtr, ytr, Xte, yte, classification):
    vals=[]; seeds=[0,1,2]
    for sd in seeds:
        m = model_fn(); h = train_one(m, Xtr, ytr, Xte, yte, classification, "adam", seed=sd, epochs=10)
        vals.append(h["val_loss"][min(4, len(h["val_loss"])-1)])
    ell = float(np.median(vals))
    opt_names = ["sgd","sgd_momentum","nesterov","adam","adam_nobias","adam_beta1zero","adamw"]
    seed_set = list(range(10))
    res={k: {"histories": [], "times_to_ell": []} for k in opt_names}
    for name in opt_names:
        for sd in seed_set:
            m = model_fn(); h = train_one(m, Xtr, ytr, Xte, yte, classification, name, seed=sd, epochs=20)
            res[name]["histories"].append(h)
            tte=None
            for i,v in enumerate(h["val_loss"]):
                if v <= ell: tte=i+1; break
            res[name]["times_to_ell"].append(tte)
    return ell, res


In [ ]:

# ▶️ Run Part 2
OUT2 = OUT_DIR / "part2_ablation"; OUT2.mkdir(parents=True, exist_ok=True)

def run_ablation(name, prep, classification, hidden=[128,64], out_dim=2, epochs=20, batch=512):
    try:
        (Xtr,ytr),(Xte,yte) = prep()
    except FileNotFoundError as e:
        print(f"Skipping {name}: {e}"); return None
    in_dim = Xtr.shape[1]
    model_fn = lambda: MLP(in_dim, hidden=hidden, out_dim=out_dim)
    h1,h2 = coarse_grids(model_fn, Xtr, ytr, Xte, yte, classification)
    ell, res = stability_and_threshold(model_fn, Xtr, ytr, Xte, yte, classification)
    with open(OUT2/f"{name}_heat_alpha_beta1.json","w") as f: json.dump(h1,f,indent=2)
    with open(OUT2/f"{name}_heat_alpha_beta2.json","w") as f: json.dump(h2,f,indent=2)
    with open(OUT2/f"{name}_stability_threshold.json","w") as f: json.dump({"ell":ell,"results":res},f,indent=2)
    print(f"[{name}] ℓ = {ell:.4f}")
    return ell

ell_hotel = run_ablation("hotel", prepare_hotel, True, [128,64], 2, epochs=20, batch=512)
ell_accid = run_ablation("accidents", prepare_accidents, False, [128,64], 1, epochs=15, batch=2048)


## Part 3 — Regularization (Adam only)

In [ ]:

def train_with_regularization(model, Xtr, ytr, Xva, yva, *,
                              classification=True, lr=1e-3, beta1=0.9, beta2=0.999, epochs=20, batch_size=512, seed=42,
                              l2_lambda=0.0, dropout_override=None, label_smoothing=0.0, input_noise_sigma=0.0, target_noise_sigma=0.0,
                              early_stop_patience=0, early_stop_min_delta=0.0):
    torch.manual_seed(seed); np.random.seed(seed)
    if dropout_override is not None:
        for m in model.modules():
            if isinstance(m, nn.Dropout): m.p = dropout_override

    Xtr_t = torch.tensor(Xtr.astype("float32"))
    Xva_t = torch.tensor(Xva.astype("float32"))
    if classification:
        ytr_t = torch.tensor(ytr.astype("int64")); yva_t = torch.tensor(yva.astype("int64"))
        crit = nn.CrossEntropyLoss(label_smoothing=float(label_smoothing) if label_smoothing else 0.0)
    else:
        ytr_t = torch.tensor(ytr.astype("float32")); yva_t = torch.tensor(yva.astype("float32"))
        crit = nn.MSELoss()

    dl = DataLoader(TensorDataset(Xtr_t,ytr_t), batch_size=batch_size, shuffle=True)
    opt = optim.Adam(model.parameters(), lr=lr, betas=(beta1,beta2))

    best_val=float("inf"); patience=0
    hist={"train_loss":[], "val_loss":[]}
    for ep in range(epochs):
        model.train(); tot=0; n=0
        for xb,yb in dl:
            opt.zero_grad()
            if input_noise_sigma>0: xb = xb + torch.randn_like(xb)*input_noise_sigma
            out = model(xb)
            if classification:
                loss = crit(out, yb)
            else:
                if target_noise_sigma>0: yb = yb + torch.randn_like(yb)*target_noise_sigma
                loss = crit(out.squeeze(), yb)
            if l2_lambda>0:
                l2 = sum((p**2).sum() for p in model.parameters() if p.requires_grad)
                loss = loss + l2_lambda * l2 / Xtr_t.shape[0]
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += float(loss.item()); n+=1
        tr=tot/max(n,1)

        model.eval()
        with torch.no_grad():
            out = model(Xva_t)
            vl = float(crit(out if classification else out.squeeze(), yva_t).item())
        hist["train_loss"].append(tr); hist["val_loss"].append(vl)

        if early_stop_patience>0:
            if vl + early_stop_min_delta < best_val:
                best_val=vl; patience=0
            else:
                patience+=1
                if patience>=early_stop_patience: break
    return hist


In [ ]:

# ▶️ Run Part 3
OUT3 = OUT_DIR / "part3_regularization"; OUT3.mkdir(parents=True, exist_ok=True)

def run_regularization(name, prep, classification, hidden=[128,64], out_dim=2, base_epochs=20, base_batch=512):
    try:
        (Xtr,ytr),(Xte,yte) = prep()
    except FileNotFoundError as e:
        print(f"Skipping {name}: {e}"); return
    in_dim = Xtr.shape[1]
    base = dict(classification=classification, lr=1e-3, beta1=0.9, beta2=0.999, epochs=base_epochs, batch_size=base_batch)

    def run_cfg(model_kwargs, **kw):
        m = MLP(in_dim, hidden=hidden, out_dim=out_dim)
        return train_with_regularization(m, Xtr, ytr, Xte, yte, **{**base, **kw})

    outs = {
        "baseline": run_cfg({}, ),
        "l2": run_cfg({}, l2_lambda=1e-4),
        "early_stop": run_cfg({}, early_stop_patience=5 if classification else 3, early_stop_min_delta=1e-4),
        "dropout": run_cfg({}, dropout_override=0.5),
    }
    if classification:
        outs["label_smooth"] = run_cfg({}, label_smoothing=0.05)
    else:
        outs["target_noise"] = run_cfg({}, target_noise_sigma=0.05)
    outs["input_noise"] = run_cfg({}, input_noise_sigma=0.01)
    # example combos
    if classification:
        outs["combo_l2+ls"] = run_cfg({}, l2_lambda=1e-4, label_smoothing=0.05)
    else:
        outs["combo_l2+input"] = run_cfg({}, l2_lambda=1e-4, input_noise_sigma=0.01)

    with open(OUT3/f"{name}_regularization.json","w") as f: json.dump(outs,f,indent=2)
    print(f"[{name}] regularization JSON saved.")

run_regularization("hotel", prepare_hotel, classification=True, hidden=[128,64], out_dim=2, base_epochs=20, base_batch=512)
run_regularization("accidents", prepare_accidents, classification=False, hidden=[256,128], out_dim=1, base_epochs=15, base_batch=2048)
